In [0]:
%sql
CREATE CONNECTION if NOT exists usgs_api_connection
TYPE HTTP
OPTIONS (
  host = 'https://earthquake.usgs.gov',
  port = '443',
  base_path = '/fdsnws/event/1',
  bearer_token = 'na'
);

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
connection_name = "usgs_api_connection"
conn = w.connections.get(name=connection_name)
print(conn)
base_url = conn.options['host']+conn.options['base_path']
print(base_url)

In [0]:
dbutils.widgets.text("catalog_name", "youtube_dev")
catalog = dbutils.widgets.get("catalog_name")

In [0]:
%py
spark.sql(f"use CATALOG {catalog}")
spark.sql(f"use SCHEMA bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS earquake_data_volumne")

In [0]:
import requests
import json
import datetime

# --------------------------------------------------
# Configuration
# --------------------------------------------------

base_url = "https://earthquake.usgs.gov/fdsnws/event/1"

volume_path = f"/Volumes/{catalog}/bronze/earquake_data_volumne"


# --------------------------------------------------
# Main API ingestion
# --------------------------------------------------

try:

    # 1. Calculate dates
    today = datetime.date.today()
    yesterday = today - datetime.timedelta(days=1)

    today_str = today.strftime("%Y-%m-%dT00:00:00")
    yesterday_str = yesterday.strftime("%Y-%m-%dT00:00:00")

    # 2. API endpoint
    url = base_url + "/query"

    # 3. API parameters
    params = {
        "format": "geojson",
        "starttime": yesterday_str,
        "endtime": today_str
    }

    print(f"Fetching earthquake data:")
    print(f"Start: {yesterday_str}")
    print(f"End:   {today_str}")

    # 4. Call API
    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    # 5. Raise exception for HTTP errors
    response.raise_for_status()

    # 6. Convert response to Python dictionary
    data = response.json()

    # 7. Create date folder
    date_folder = yesterday.strftime("%Y-%m-%d")

    folder_path = f"{volume_path}/{date_folder}"

    dbutils.fs.mkdirs(folder_path)

    # 8. Create JSON file path
    file_path = f"{folder_path}/earthquakes.json"

    # 9. Save JSON to Volume
    dbutils.fs.put(
        file_path,
        json.dumps(data),
        overwrite=True
    )

    # 10. Success message
    print("--------------------------------------------------")
    print("API ingestion successful")
    print(f"Status code : {response.status_code}")
    print(f"File saved  : {file_path}")
    print("--------------------------------------------------")


# --------------------------------------------------
# Error handling
# --------------------------------------------------
except requests.exceptions.RequestException as e:
    print(f"API request failed: {e}")

except Exception as e:
    print(f"Unexpected error: {e}")